In [49]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import sys

sys.path.append(r"D:/Football Project/src")

import feature_lists as fl

In [50]:
df_26=pd.read_csv("../../data/processed/major_leagues/Sofascore_player_data_2526.csv")
df_25=pd.read_csv("../../data/processed/major_leagues/Sofascore_player_data_2425.csv")
df_24=pd.read_csv("../../data/processed/major_leagues/Sofascore_player_data_2324.csv")

In [51]:
df_26.columns.to_list()

['accuratechippedpasses',
 'accuratecrosses',
 'accuratecrossespercentage',
 'accuratefinalthirdpasses',
 'accuratelongballs',
 'accuratelongballspercentage',
 'accurateoppositionhalfpasses',
 'accurateownhalfpasses',
 'accuratepasses',
 'accuratepassespercentage',
 'aerialduelswon',
 'aerialduelswonpercentage',
 'aeriallost',
 'appearances',
 'assists',
 'attemptpenaltymiss',
 'attemptpenaltypost',
 'attemptpenaltytarget',
 'ballrecovery',
 'bigchancescreated',
 'bigchancesmissed',
 'blockedshots',
 'cleansheet',
 'clearances',
 'countrating',
 'crossesnotclaimed',
 'directredcards',
 'dispossessed',
 'dribbledpast',
 'duellost',
 'errorleadtogoal',
 'errorleadtoshot',
 'expectedassists',
 'expectedgoals',
 'fouls',
 'freekickgoal',
 'goalconversionpercentage',
 'goalkicks',
 'goals',
 'goalsassistssum',
 'goalsconceded',
 'goalsconcededinsidethebox',
 'goalsconcededoutsidethebox',
 'goalsfrominsidethebox',
 'goalsfromoutsidethebox',
 'groundduelswon',
 'groundduelswonpercentage',
 'h

In [52]:
df_26.shape

(4872, 402)

In [ ]:
df_2024 = df_24.drop_duplicates(subset=["player id", "season"], keep="last")
df_2025 = df_25.drop_duplicates(subset=["player id", "season"], keep="last")
df_2026 = df_26.drop_duplicates(subset=["player id", "season"], keep="last")

active_2026_player_ids = set(df_2026["player id"].unique())

df_list = [df_2024, df_2025, df_2026]
df_all_seasons = pd.concat(df_list, ignore_index=True)
df_all_seasons = df_all_seasons[df_all_seasons["player id"].isin(active_2026_player_ids)]
df_all_seasons = df_all_seasons.sort_values(by=["player id", "season"]).reset_index(drop=True)

performance_zscore_cols = list(set(
    fl.per90_zscore_cols + 
    fl.volume_zscore_cols + 
    fl.rate_zscore_cols
))
performance_zscore_cols = [col for col in performance_zscore_cols if col in df_all_seasons.columns]

availability_tracking = ["appearances", "matchesstarted", "minutesplayed"]
all_tracking_metrics = performance_zscore_cols + availability_tracking

int_cols = df_all_seasons[all_tracking_metrics].select_dtypes(include=[np.int64, np.int32]).columns
if len(int_cols) > 0:
    df_all_seasons = df_all_seasons.astype({col: "float64" for col in int_cols})

df_meta_latest = df_all_seasons.groupby("player id").last().reset_index()
df_season_counts = df_all_seasons.groupby("player id")["season"].nunique().reset_index(name="number_of_seasons")

df_base_identity = pd.merge(
    df_meta_latest[["player id", "player", "team id", "team", "league", "position"]], 
    df_season_counts, 
    on="player id"
)

df_avg_calculations = df_all_seasons.groupby("player id")[all_tracking_metrics].mean().reset_index()
df_avg = pd.merge(df_base_identity, df_avg_calculations, on="player id")

def fast_ewma(series, alpha=0.5):
    n = len(series)
    if n == 1:
        return series.iloc[0]
    weights = (1 - alpha) ** np.arange(n)[::-1]
    return np.sum(series * weights) / np.sum(weights)

df_ewma_calculations = df_all_seasons.groupby("player id")[all_tracking_metrics].agg(fast_ewma).reset_index()
df_ewma = pd.merge(df_base_identity, df_ewma_calculations, on="player id")

for col in fl.rate_cols:
    z_col = f"{col}_zscore"
    if z_col in df_avg.columns:
        df_avg[z_col] = df_avg[z_col].fillna(0)
    if z_col in df_ewma.columns:
        df_ewma[z_col] = df_ewma[z_col].fillna(0)

clean_metadata_order = [
    "player id", "player", "team id", "team", "league", "position", 
    "number_of_seasons", "appearances", "matchesstarted", "minutesplayed"
]

df_avg = df_avg[clean_metadata_order + performance_zscore_cols].copy()
df_ewma = df_ewma[clean_metadata_order + performance_zscore_cols].copy()

C:\Users\vibha\AppData\Local\Temp\ipykernel_11972\2750717383.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_meta_latest = df_all_seasons.groupby("player id").last().reset_index()
C:\Users\vibha\AppData\Local\Temp\ipykernel_11972\2750717383.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_avg_calculations = df_all_seasons.groupby("player id")[all_tracking_metrics].mean().reset_index()


⚡ Building separate structural matrices...
✅ SUCCESS: Separate tracking tables finalized with no data noise.


C:\Users\vibha\AppData\Local\Temp\ipykernel_11972\2750717383.py:47: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ewma_calculations = df_all_seasons.groupby("player id")[all_tracking_metrics].agg(fast_ewma).reset_index()


In [56]:
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(df_ewma.sample(10))

,player id,player,team id,team,league,position,number_of_seasons,appearances,matchesstarted,minutesplayed,...,directredcards_zscore,accurateownhalfpasses_per90_zscore,penaltiestaken_per90_zscore,savescaught_per90_zscore,successfuldribbles_zscore,goals_zscore,totalcontest_zscore,accurateoppositionhalfpasses_per90_zscore,errorleadtogoal_per90_zscore,totalduelswon_zscore
1705,848385,Dario Šarić,3056,Antalyaspor,Turkiye Super Lig,Midfielders,3,24.857143,17.142857,1482.142857,...,-0.327215,-0.458544,-0.334925,0.000000,-0.343291,-0.090740,0.105261,-0.889570,-0.399031,0.477448
1555,833057,Hamza Choudhury,31,Leicester City,England EFL Championship,Midfielders,3,26.571429,15.571429,1400.142857,...,-0.213659,1.688070,-0.278268,0.000000,-0.967747,-0.945113,-1.003527,0.359141,-0.411730,-0.661532
1243,798583,Dayot Upamecano,2672,FC Bayern München,Germany Bundesliga,Defenders,3,23.000000,20.428571,1782.571429,...,-0.284194,1.668613,-0.207516,0.000000,-0.537160,0.016032,-0.412173,2.711331,0.097208,-0.148076
2978,987650,Moisés Caicedo,38,Chelsea,England Premier League,Midfielders,3,34.714286,33.857143,2970.285714,...,2.019172,1.963881,-0.276983,0.000000,-0.057455,-0.294897,-0.196972,1.333839,0.314938,1.640348
4121,1160559,Oliver Provstgaard,2699,Lazio,Italy Serie A,Defenders,1,26.000000,13.000000,1544.000000,...,-0.340451,1.192002,-0.097590,0.000000,-0.778024,-0.880217,-0.757695,1.466136,-0.739073,-0.981210
1597,839410,Aaron Ramsdale,39,Newcastle United,England Premier League,Goalkeepers,2,18.000000,17.333333,1568.666667,...,-0.205588,0.232296,0.000000,-0.044086,-0.534125,0.000000,-0.534125,0.380512,-1.159619,-1.044317
1971,877201,Gabriel Strefezza,2690,Parma,Italy Serie A,Forwards,3,21.285714,18.428571,1659.857143,...,-0.097074,0.606451,-0.574423,-0.008885,0.253476,-0.672733,0.033404,1.268567,-0.269145,-0.022501
805,352370,Lucas Hernández,1644,Paris Saint-Germain,France Ligue 1,Defenders,2,25.666667,21.000000,1805.666667,...,-0.461629,1.158363,-0.170582,0.000000,-0.540320,-0.494731,-0.450450,2.562196,-0.608608,-0.206100
2237,909479,David Martínez,36839,Defensa y Justicia,Argentina Liga Profesional,Defenders,1,12.000000,12.000000,933.000000,...,-0.300000,0.468377,-0.189568,0.000000,-0.158064,-0.682556,0.014209,1.345602,-0.323222,-1.506862
3395,1020500,Matías Fernández,36842,Independiente Rivadavia,Argentina Liga Profesional,Midfielders,1,15.000000,14.000000,1040.000000,...,-0.263621,-0.985940,-0.260532,0.000000,0.008616,0.723569,0.863246,-0.350179,-0.229360,-1.353360


In [59]:
df_ewma[df_ewma['player']=='Lamine Yamal']

,player id,player,team id,team,league,position,number_of_seasons,appearances,matchesstarted,minutesplayed,...,directredcards_zscore,accurateownhalfpasses_per90_zscore,penaltiestaken_per90_zscore,savescaught_per90_zscore,successfuldribbles_zscore,goals_zscore,totalcontest_zscore,accurateoppositionhalfpasses_per90_zscore,errorleadtogoal_per90_zscore,totalduelswon_zscore
4416,1402912,Lamine Yamal,2817,FC Barcelona,Spain La Liga,Midfielders,3,31.285714,26.857143,2427.285714,...,-0.37861,-0.999407,2.269481,0.0,6.545872,3.558039,5.872316,1.414109,-0.419128,2.847429


In [60]:
df_avg[df_avg['player']=='Lamine Yamal']

,player id,player,team id,team,league,position,number_of_seasons,appearances,matchesstarted,minutesplayed,...,directredcards_zscore,accurateownhalfpasses_per90_zscore,penaltiestaken_per90_zscore,savescaught_per90_zscore,successfuldribbles_zscore,goals_zscore,totalcontest_zscore,accurateoppositionhalfpasses_per90_zscore,errorleadtogoal_per90_zscore,totalduelswon_zscore
4416,1402912,Lamine Yamal,2817,FC Barcelona,Spain La Liga,Midfielders,3,33.333333,26.333333,2442.0,...,-0.364853,-1.052961,1.180849,0.0,5.754754,2.628555,5.168054,1.214594,-0.404303,2.646606


In [62]:
df_ewma.to_csv('../../data/processed/major_leagues/EWMA_Major_Leagues.csv',index=False)
df_avg.to_csv('../../data/processed/major_leagues/Avg_Major_Leagues.csv',index=False)

### Data has been merged similar to how it was done for V1 ###